# Task 6 — Query Caching Layer
## Project 20: Distributed Reverse Image Search Engine — Milestone 2
### Author: Aliza

---

## ⚠️ EXECUTION ENVIRONMENT — READ BEFORE RUNNING

| Requirement | Kaggle | Local |
|---|---|---|
| Task 6 code (LRUCache, CachedSearchEngine) | ✅ Runs fine | ✅ Runs fine |
| Unit tests (eviction, standalone) | ✅ Runs fine | ✅ Runs fine |
| Demo mode (600 test images, repo data only) | ✅ Runs fine | ✅ Runs fine |
| Full system integration (1M images) | ❌ **Cannot run** | ✅ Required |

### ❌ This notebook MUST be run locally for full integration

**Why it cannot run on Kaggle:**
1. `all_cnn_embeddings.h5` (~512 MB), `all_perceptual_hashes.pkl` (~80 MB), `all_sift_features.pkl`, `all_orb_features.pkl`, `all_hist_features.pkl`, `all_image_paths.pkl` — all live on the **team's Google Drive**, not as public Kaggle datasets.
2. The full LSH index (`table_0.pkl` … `table_9.pkl`, ~500–700 MB total for 1M images) was built by Piranchal locally and shared via Drive.
3. `SearchEngine` (Mahnoor's Task 4) loads ~1.5 GB of data at startup — this data is not available in the Kaggle environment.
4. `multiprocessing` (used internally by `build_index.py`) does not work reliably in Kaggle's sandboxed environment.

**To run this notebook fully locally:**
```
project_root/
├── data/
│   ├── all_cnn_embeddings.h5          ← from Google Drive
│   ├── all_perceptual_hashes.pkl       ← your M1 output
│   ├── all_sift_features.pkl           ← from Google Drive
│   ├── all_orb_features.pkl            ← from Google Drive
│   ├── all_hist_features.pkl           ← from Google Drive
│   └── all_image_paths.pkl             ← from Google Drive
├── lsh_index/                          ← Piranchal's Task 1+2 output
│   ├── lsh_config.json
│   ├── projection_matrices.pkl
│   └── hash_tables/ (table_0.pkl … table_9.pkl)
├── feature_Fusion/feature_fusion.py
├── query_engine/
│   ├── query_processor.py
│   └── search_engine.py
└── validation/
    ├── ground_truth.json
    └── test_features/
```

---
**This notebook is organised in three parts:**
- **Part A** — Full Task 6 implementation (all classes, runs everywhere)
- **Part B** — Unit tests + Demo mode (uses 600 test images, runs without the 1M dataset)
- **Part C** — Full integration test (requires local data + full SearchEngine)


In [1]:
# ── CELL 1: Install dependencies ─────────────────────────────────────────────
# All should already be present from Milestone 1 except imagehash.
# Uncomment the line below if running in a fresh environment:
#!pip install numpy scipy scikit-learn h5py tqdm imagehash Pillow --quiet

import importlib, sys
missing = [p for p in ['numpy','h5py','imagehash','PIL','tqdm']
           if importlib.util.find_spec(p) is None]
if missing:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                           'numpy', 'h5py', 'imagehash', 'Pillow', 'tqdm', '--quiet'])

print('✓ All dependencies available.')

✓ All dependencies available.


In [8]:
pip install "numpy>=2.0" --upgrade

   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
    --------------------------------------- 0.3/12.3 MB ? eta -:--:--
   --- ------------------------------------ 1.0/12.3 MB 2.8 MB/s eta 0:00:05
   ------ --------------------------------- 2.1/12.3 MB 4.3 MB/s eta 0:00:03
   ------- -------------------------------- 2.4/12.3 MB 3.9 MB/s eta 0:00:03
   -------- ------------------------------- 2.6/12.3 MB 2.6 MB/s eta 0:00:04
   ---------- ----------------------------- 3.1/12.3 MB 2.6 MB/s eta 0:00:04
   ------------ --------------------------- 3.9/12.3 MB 2.7 MB/s eta 0:00:04
   -------------- ------------------------- 4.5/12.3 MB 2.7 MB/s eta 0:00:03
   -------------- ------------------------- 4.5/12.3 MB 2.7 MB/s eta 0:00:03
   ---------------- ----------------------- 5.0/12.3 MB 2.4 MB/s eta 0:00:04
   ------------------ --------------------- 5.8/12.3 MB 2.6 MB/s eta 0:00:03
   -----------------

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
face-recognition 1.3.0 requires dlib>=19.7, which is not installed.


In [2]:
# ── CELL 2: Imports ───────────────────────────────────────────────────────────
import os
import sys
import json
import pickle
import time
import collections
import numpy as np
import h5py
from tqdm import tqdm
from PIL import Image
import imagehash

print('✓ Imports complete.')

✓ Imports complete.


In [22]:
import os

print(os.listdir('/kaggle/input'))
print(os.listdir('/kaggle/input/datasets/wari30/pdc-results'))

['datasets']
['PDC_distributed-reverse-image-search']


In [23]:
# ── CELL 3: Path configuration ────────────────────────────────────────────────

import os, sys, json

REPO_ROOT = '/kaggle/input/datasets/wari30/pdc-results/PDC_distributed-reverse-image-search'

sys.path.insert(0, REPO_ROOT)

INDEX_DIR         = os.path.join(REPO_ROOT, 'lsh_index')
DATA_DIR          = os.path.join(REPO_ROOT, 'data')           # not present on Kaggle — Part C skips
CACHE_OUTPUT_DIR  = '/kaggle/working/cache'                   # writable output dir
TEST_FEATURES_DIR = os.path.join(REPO_ROOT, 'validation', 'test_features')
GROUND_TRUTH_PATH = os.path.join(REPO_ROOT, 'validation', 'ground_truth.json')
LSH_CONFIG_PATH   = os.path.join(INDEX_DIR, 'lsh_config.json')

os.makedirs(CACHE_OUTPUT_DIR, exist_ok=True)

with open(LSH_CONFIG_PATH) as f:
    lsh_cfg = json.load(f)

CACHE_MAX_SIZE          = lsh_cfg['cache_max_size']          # 1000
CACHE_HAMMING_THRESHOLD = lsh_cfg['cache_hamming_threshold'] # 8
TOP_K                   = lsh_cfg['top_k']                   # 10
HASH_SIZE               = 8

FULL_DATA_AVAILABLE = False   # data/ not on Kaggle — Part C will be skipped

print('=' * 60)
print(f'REPO_ROOT          : {REPO_ROOT}')
print(f'LSH config         : {lsh_cfg}')
print(f'Test features dir  : {TEST_FEATURES_DIR}')
print(f'Cache output dir   : {CACHE_OUTPUT_DIR}')
print(f'Full data avail.   : {FULL_DATA_AVAILABLE}')
print('  → DEMO MODE: Parts A, B, D run fully.')
print('    Part C will be SKIPPED (expected on Kaggle).')
print('=' * 60)

REPO_ROOT          : /kaggle/input/datasets/wari30/pdc-results/PDC_distributed-reverse-image-search
LSH config         : {'num_tables': 10, 'hash_size': 12, 'embedding_dim': 128, 'random_seed': 42, 'top_k': 10, 'max_candidates': 5000, 'cache_max_size': 1000, 'cache_hamming_threshold': 8}
Test features dir  : /kaggle/input/datasets/wari30/pdc-results/PDC_distributed-reverse-image-search/validation/test_features
Cache output dir   : /kaggle/working/cache
Full data avail.   : False
  → DEMO MODE: Parts A, B, D run fully.
    Part C will be SKIPPED (expected on Kaggle).


In [ ]:
# ── CELL 3: Path configuration — LOCAL RUN ────────────────────────────────────

import os, sys, json

# ── Paths based on your local machine ─────────────────────────────────────────
# Notebook location : C:\Users\USER\Desktop\pdc proj\pdc\pdc-milestone2-task6.ipynb
# Repo location     : C:\Users\USER\Desktop\pdc proj\pdc\PDC_distributed-reverse-image-search\
#                     (extract your .rar here first)

REPO_ROOT = r'C:\Users\USER\Desktop\pdc proj\pdc\PDC_distributed-reverse-image-search'

sys.path.insert(0, REPO_ROOT)

INDEX_DIR         = os.path.join(REPO_ROOT, 'lsh_index')
DATA_DIR          = os.path.join(REPO_ROOT, 'data')          # put all merged M1 files here
CACHE_OUTPUT_DIR  = os.path.join(REPO_ROOT, 'cache')
TEST_FEATURES_DIR = os.path.join(REPO_ROOT, 'validation', 'test_features')
GROUND_TRUTH_PATH = os.path.join(REPO_ROOT, 'validation', 'ground_truth.json')
LSH_CONFIG_PATH   = os.path.join(INDEX_DIR, 'lsh_config.json')

os.makedirs(CACHE_OUTPUT_DIR, exist_ok=True)

with open(LSH_CONFIG_PATH) as f:
    lsh_cfg = json.load(f)

CACHE_MAX_SIZE          = lsh_cfg['cache_max_size']          # 1000
CACHE_HAMMING_THRESHOLD = lsh_cfg['cache_hamming_threshold'] # 8
TOP_K                   = lsh_cfg['top_k']                   # 10
HASH_SIZE               = 8

# Check which M1 feature files are present in data/
_required = ['all_cnn_embeddings.h5', 'all_perceptual_hashes.pkl',
             'all_sift_features.pkl', 'all_orb_features.pkl',
             'all_hist_features.pkl', 'all_image_paths.pkl']

FULL_DATA_AVAILABLE = all(os.path.exists(os.path.join(DATA_DIR, f)) for f in _required)

print('=' * 60)
print(f'REPO_ROOT          : {REPO_ROOT}')
print(f'LSH config         : {lsh_cfg}')
print(f'Test features dir  : {TEST_FEATURES_DIR}')
print(f'Cache output dir   : {CACHE_OUTPUT_DIR}')
print(f'Full data avail.   : {FULL_DATA_AVAILABLE}')
print()
print('data/ folder status:')
for f in _required:
    fpath = os.path.join(DATA_DIR, f)
    status = '✅' if os.path.exists(fpath) else '❌ MISSING'
    size   = f'  ({os.path.getsize(fpath)/1e6:.0f} MB)' if os.path.exists(fpath) else ''
    print(f'  {status}  {f}{size}')
print()
if FULL_DATA_AVAILABLE:
    print('→ FULL MODE: Parts A, B, C, D will all run.')
else:
    print('→ DEMO MODE: Parts A, B, D run. Part C skipped.')
    print('  Place merged M1 files into:')
    print(f'  {DATA_DIR}')
print('=' * 60)

REPO_ROOT          : C:\Users\USER\Desktop\pdc proj\pdc\PDC_distributed-reverse-image-search
LSH config         : {'num_tables': 10, 'hash_size': 12, 'embedding_dim': 128, 'random_seed': 42, 'top_k': 10, 'max_candidates': 5000, 'cache_max_size': 1000, 'cache_hamming_threshold': 8}
Test features dir  : C:\Users\USER\Desktop\pdc proj\pdc\PDC_distributed-reverse-image-search\validation\test_features
Cache output dir   : C:\Users\USER\Desktop\pdc proj\pdc\PDC_distributed-reverse-image-search\cache
Full data avail.   : True

data/ folder status:
  ✅  all_cnn_embeddings.h5  (482 MB)
  ✅  all_perceptual_hashes.pkl  (55 MB)
  ✅  all_sift_features.pkl  (545 MB)
  ✅  all_orb_features.pkl  (161 MB)
  ✅  all_hist_features.pkl  (417 MB)
  ✅  all_image_paths.pkl  (85 MB)

→ FULL MODE: Parts A, B, C, D will all run.


---
## Part A — Task 6 Full Implementation
### A1 — Core Hash Utilities

In [4]:
# ── CELL 4: Hamming distance utilities ───────────────────────────────────────
# Copied exactly from Aliza's Milestone 1 hashes.ipynb (Cell 9) for consistency.

def hamming_distance(h1: int, h2: int) -> int:
    """
    Number of bit positions that differ between two hash integers.
    Uses XOR + popcount: positions that differ become 1-bits.
    """
    return bin(h1 ^ h2).count('1')


def hash_similarity(h1: int, h2: int) -> float:
    """
    Hash similarity in [0, 1]. 1 = identical hashes, 0 = all bits differ.
    Based on 64-bit hashes (hash_size=8 → 8×8 = 64 bits).
    """
    return 1.0 - (hamming_distance(h1, h2) / 64.0)


# ── Quick smoke test ──────────────────────────────────────────────────────────
assert hamming_distance(0, 0) == 0,          "identical hashes must have distance 0"
assert hamming_distance(0, 0xFFFFFFFFFFFFFFFF) == 64, "all bits differ = distance 64"
assert hamming_distance(0b1010, 0b1100) == 2, "2 bits differ"
assert hash_similarity(0, 0) == 1.0,          "identical = similarity 1.0"
assert hash_similarity(0, 0xFFFFFFFFFFFFFFFF) == 0.0, "opposite = similarity 0.0"

print('✓ hamming_distance() and hash_similarity() verified.')

✓ hamming_distance() and hash_similarity() verified.


### A2 — LRU Cache Data Structure

In [5]:
# ── CELL 5: LRUCache ─────────────────────────────────────────────────────────

class LRUCache:
    """
    Least-Recently-Used cache for query results.

    Internal storage: collections.OrderedDict
        key   → integer pHash
        value → (result_list, access_timestamp)

    LRU policy: most-recently accessed/inserted entry sits at the END of the
    OrderedDict.  On overflow, the entry at the FRONT (LRU) is evicted.

    Public API
    ----------
    get(phash_key)                      – O(1) exact lookup
    put(phash_key, result_list)         – O(1) insert with eviction
    find_approximate(query_phash, thr)  – O(cache_size) nearest-hash scan
    clear()                             – wipe all entries
    __len__()                           – current number of cached entries
    """

    def __init__(self, max_size: int):
        if max_size < 1:
            raise ValueError('max_size must be >= 1')
        self._max_size  = max_size
        self._store     = collections.OrderedDict()   # phash_int → (result_list, ts)

    # ── Exact lookup ──────────────────────────────────────────────────────────
    def get(self, phash_key: int):
        """
        Return result_list for phash_key if present, else None.
        Moves the entry to the back (MRU position) on access.
        """
        if phash_key not in self._store:
            return None
        result_list, _ = self._store[phash_key]
        # Move to MRU position
        self._store.move_to_end(phash_key)
        self._store[phash_key] = (result_list, time.time())
        return result_list

    # ── Insert with LRU eviction ──────────────────────────────────────────────
    def put(self, phash_key: int, result_list: list):
        """
        Insert phash_key → result_list.
        Evicts the LRU entry (front of OrderedDict) if at capacity.
        """
        if phash_key in self._store:
            # Update existing entry, move to MRU
            self._store.move_to_end(phash_key)
        self._store[phash_key] = (result_list, time.time())
        if len(self._store) > self._max_size:
            # Evict LRU (front of OrderedDict — first inserted / least accessed)
            evicted_key, _ = self._store.popitem(last=False)
            # (optional: log eviction for debugging)

    # ── Approximate match (Level-2 lookup) ───────────────────────────────────
    def find_approximate(self, query_phash: int, threshold: int):
        """
        Scan all cached keys for the one with the smallest Hamming distance
        to query_phash.  Return its result_list if distance ≤ threshold.

        Tie-breaking: if multiple keys share the minimum distance, return
        the one that was most recently accessed (= rightmost in OrderedDict).

        Complexity: O(cache_size) — at most O(1000) with default max_size.

        Why pHash only (not aHash/dHash):
            pHash is DCT-based and more robust to minor visual changes than
            aHash/dHash.  Using a single hash type avoids ambiguity when the
            three types disagree.

        Returns result_list or None.
        """
        best_key  = None
        best_dist = threshold + 1   # sentinel: no valid match yet

        # Iterate most-recent first so MRU wins ties
        for cached_key, (result_list, _) in reversed(self._store.items()):
            dist = hamming_distance(query_phash, cached_key)
            if dist < best_dist:   # strict '<' keeps MRU as tie-breaker
                best_dist = dist
                best_key  = cached_key

        if best_key is not None and best_dist <= threshold:
            result_list, _ = self._store[best_key]
            # Move matched entry to MRU
            self._store.move_to_end(best_key)
            self._store[best_key] = (result_list, time.time())
            return result_list

        return None

    # ── Utility ───────────────────────────────────────────────────────────────
    def clear(self):
        """Wipe all entries (call this whenever the LSH index is rebuilt)."""
        self._store.clear()

    def __len__(self) -> int:
        return len(self._store)

    def __repr__(self) -> str:
        return f'LRUCache(size={len(self)}/{self._max_size})'


print('✓ LRUCache defined.')

✓ LRUCache defined.


### A3 — CachedSearchEngine

In [6]:
# ── CELL 6: CachedSearchEngine ────────────────────────────────────────────────

class CachedSearchEngine:
    """
    Caching layer that wraps any SearchEngine-compatible object.

    Cache Key Strategy
    ------------------
    Keys are pHash integers (64-bit).  Two queries are considered identical
    if their pHash Hamming distance is ≤ CACHE_HAMMING_THRESHOLD (8 bits),
    which corresponds to minor visual differences: brightness shifts, light
    JPEG recompression, slight crops.

    Two-level lookup:
      Level 1 — exact match O(1): direct dict key lookup.
      Level 2 — approximate match O(1000): linear scan over cached keys.

    Three input types:
      1. query_image_id  (int)         – in-dataset image, pHash from lookup table
      2. query_pil_image (PIL.Image)   – new image, pHash computed on-the-fly
      3. query_embedding (np.ndarray)  – pre-extracted embedding, no pHash
                                         available → bypasses cache entirely

    Args
    ----
    search_engine   : any object with a .search(embedding, image_id, top_k) method
                      that returns list[(image_id, score)] sorted descending
    phash_lookup    : dict {global_id: {'ahash': int, 'dhash': int, 'phash': int}}
                      — Aliza's all_perceptual_hashes.pkl from Milestone 1
    max_size        : LRU cache capacity (default: lsh_config CACHE_MAX_SIZE)
    hamming_thresh  : approximate-match threshold in bits (default: 8)
    """

    def __init__(self,
                 search_engine,
                 phash_lookup: dict,
                 max_size: int          = CACHE_MAX_SIZE,
                 hamming_thresh: int    = CACHE_HAMMING_THRESHOLD):

        self.search_engine   = search_engine
        self.phash_lookup    = phash_lookup          # global_id → {ahash, dhash, phash}
        self.hamming_thresh  = hamming_thresh
        self.cache           = LRUCache(max_size=max_size)

        # ── Counters ──────────────────────────────────────────────────────
        self.total_queries    = 0
        self.cache_hits       = 0
        self.exact_hits       = 0
        self.approximate_hits = 0
        self.cache_misses     = 0
        self.cache_bypasses   = 0   # queries routed around the cache

        print(f'✓ CachedSearchEngine ready.  '
              f'Cache size={max_size}, hamming_thresh={hamming_thresh}')

    # =========================================================================
    # Main entry point
    # =========================================================================

    def cached_search(self,
                      query_image_id: int       = None,
                      query_embedding           = None,
                      query_pil_image: Image.Image = None,
                      top_k: int                = TOP_K) -> list:
        """
        Search with caching.  Accepts three mutually exclusive input types.

        Input type 1 — query_image_id (int):
            Image is in the MIRFLICKR dataset.  Looks up its pHash from
            self.phash_lookup, then does a two-level cache check.  On miss,
            calls search_engine.search(embedding, image_id=query_image_id).

        Input type 2 — query_embedding (numpy array, shape (128,)):
            Pre-extracted embedding for a new image not in the dataset.
            No pHash available → BYPASSES the cache entirely and calls
            search_engine.search(embedding, image_id=None) directly.
            This logs a 'cache_bypass' event.

        Input type 3 — query_pil_image (PIL.Image):
            New image not in the dataset.  Computes pHash on-the-fly using
            imagehash.phash(), then does the two-level cache check.

        Trade-off note (approximate hits):
            An approximate hit returns a result computed for a slightly
            different image.  This is intentional: images differing by
            ≤8 Hamming bits are visually near-identical (brightness shift,
            mild JPEG), so their top-10 similar results are essentially
            the same.  This was validated in Milestone 1's test set.

        Returns
        -------
        list of (image_id: int, score: float), sorted descending by score.
        """
        # ── Validate inputs ───────────────────────────────────────────────
        n_inputs = sum(x is not None for x in
                       [query_image_id, query_embedding, query_pil_image])
        if n_inputs == 0:
            raise ValueError('Provide exactly one of: query_image_id, '
                             'query_embedding, or query_pil_image.')
        if n_inputs > 1:
            raise ValueError('Provide exactly ONE input type, not multiple.')

        self.total_queries += 1

        # ─────────────────────────────────────────────────────────────────
        # INPUT TYPE 2 — raw embedding — bypass cache
        # ─────────────────────────────────────────────────────────────────
        if query_embedding is not None:
            self.cache_bypasses += 1
            embedding = np.array(query_embedding, dtype=np.float32).flatten()
            norm = np.linalg.norm(embedding)
            if norm > 1e-8:
                embedding = embedding / norm
            return self.search_engine.search(
                query_embedding=embedding,
                query_image_id=None,
                top_k=top_k
            )

        # ─────────────────────────────────────────────────────────────────
        # INPUT TYPE 3 — PIL Image — compute pHash on-the-fly
        # ─────────────────────────────────────────────────────────────────
        if query_pil_image is not None:
            ph_obj       = imagehash.phash(query_pil_image, hash_size=HASH_SIZE)
            query_phash  = int(str(ph_obj), 16)
            # Need an embedding too — derive from imagehash? No, we need a CNN
            # embedding. For PIL image queries without a pre-extracted embedding,
            # we must check the cache first and only compute embedding on miss.
            return self._lookup_and_search(
                query_phash    = query_phash,
                embedding_fn   = lambda: self._embed_pil(query_pil_image),
                image_id       = None,
                top_k          = top_k
            )

        # ─────────────────────────────────────────────────────────────────
        # INPUT TYPE 1 — image_id — look up pHash from dataset
        # ─────────────────────────────────────────────────────────────────
        if query_image_id not in self.phash_lookup:
            raise KeyError(f'image_id {query_image_id} not found in phash_lookup. '
                           f'Only IDs present in all_perceptual_hashes.pkl are supported.')

        query_phash = self.phash_lookup[query_image_id]['phash']
        return self._lookup_and_search(
            query_phash  = query_phash,
            embedding_fn = lambda: self._get_embedding_for_id(query_image_id),
            image_id     = query_image_id,
            top_k        = top_k
        )

    # =========================================================================
    # Internal: two-level cache lookup + search-on-miss
    # =========================================================================

    def _lookup_and_search(self, query_phash, embedding_fn, image_id, top_k):
        """
        Perform the two-level cache lookup.  Call embedding_fn() + search
        only on a cache miss to avoid expensive embedding loading when possible.
        """
        # ── Level 1: Exact match (O(1)) ───────────────────────────────────
        cached = self.cache.get(query_phash)
        if cached is not None:
            self.cache_hits  += 1
            self.exact_hits  += 1
            return cached

        # ── Level 2: Approximate match (O(cache_size)) ────────────────────
        approx = self.cache.find_approximate(query_phash, self.hamming_thresh)
        if approx is not None:
            self.cache_hits       += 1
            self.approximate_hits += 1
            # Also register this query_phash as a key so future exact queries
            # on the same hash hit Level 1 directly.
            self.cache.put(query_phash, approx)
            return approx

        # ── Cache miss: run full search ───────────────────────────────────
        self.cache_misses += 1
        embedding = embedding_fn()              # load/compute CNN embedding now
        result    = self.search_engine.search(
            query_embedding = embedding,
            query_image_id  = image_id,
            top_k           = top_k
        )
        self.cache.put(query_phash, result)     # store for future queries
        return result

    # =========================================================================
    # Helpers
    # =========================================================================

    def _get_embedding_for_id(self, image_id: int) -> np.ndarray:
        """
        Delegate to the underlying search engine's query processor.
        Returns normalized (128,) float32 embedding.
        """
        return self.search_engine.query_processor.prepare_query_embedding(
            image_id=image_id
        )

    def _embed_pil(self, pil_image: Image.Image) -> np.ndarray:
        """
        Extract a CNN embedding from a raw PIL image for use when only a PIL
        image was provided (Input type 3).  Requires the search engine to
        expose a FeatureStore with a get_cnn() method OR a CNN model for
        on-the-fly inference.  Falls back to a zero vector with a warning
        if neither is available (unlikely in practice — the caller should
        provide query_image_id or query_embedding for efficiency).
        """
        # Try to use the search engine's feature store if it's been loaded
        if hasattr(self.search_engine, 'feature_store'):
            raise NotImplementedError(
                'On-the-fly CNN embedding extraction from a PIL image requires '
                'a loaded CNN model (e.g. ResNet-50).  For PIL image queries, '
                'pre-extract the embedding externally and use query_embedding= instead.'
            )
        raise NotImplementedError('PIL-based embedding extraction not implemented.')

    # =========================================================================
    # Cache management
    # =========================================================================

    def warm_cache(self, image_ids: list, top_k: int = TOP_K):
        """
        Pre-populate the cache with results for a list of image IDs.
        Runs full searches (cache misses) and stores results so future
        users get instant responses.

        Useful for pre-loading results for frequently queried images
        before the system goes live.
        """
        print(f'Warming cache with {len(image_ids)} images ...')
        for image_id in tqdm(image_ids, desc='Warm-up', unit='img'):
            try:
                self.cached_search(query_image_id=image_id, top_k=top_k)
            except Exception as e:
                print(f'  Warning: warm_cache skipped ID {image_id}: {e}')
        print(f'Cache warmed.  Current size: {len(self.cache)}')

    # =========================================================================
    # Statistics
    # =========================================================================

    def get_cache_stats(self) -> dict:
        """
        Return a snapshot of all cache performance counters.

        estimated_time_saved_ms uses the search engine's mean latency so the
        number is calibrated to actual query runtime, not an arbitrary constant.
        """
        hit_rate = (self.cache_hits / self.total_queries * 100.0
                    if self.total_queries > 0 else 0.0)

        # Get mean search latency from the underlying search engine
        try:
            se_stats  = self.search_engine.get_search_stats()
            mean_ms   = se_stats.get('mean_latency_ms', 0.0)
        except Exception:
            mean_ms   = 0.0

        return {
            'total_queries':          self.total_queries,
            'cache_hits':             self.cache_hits,
            'cache_misses':           self.cache_misses,
            'exact_hits':             self.exact_hits,
            'approximate_hits':       self.approximate_hits,
            'cache_bypasses':         self.cache_bypasses,
            'hit_rate_pct':           round(hit_rate, 2),
            'current_cache_size':     len(self.cache),
            'max_cache_size':         self.cache._max_size,
            'mean_search_latency_ms': round(mean_ms, 2),
            'estimated_time_saved_ms': round(self.cache_hits * mean_ms, 2),
        }

    def save_cache_stats(self, output_path: str = None):
        """
        Write get_cache_stats() to a JSON file.
        Default path: CACHE_OUTPUT_DIR/cache_stats.json
        """
        if output_path is None:
            output_path = os.path.join(CACHE_OUTPUT_DIR, 'cache_stats.json')
        stats = self.get_cache_stats()
        with open(output_path, 'w') as f:
            json.dump(stats, f, indent=2)
        print(f'✓ Cache stats saved to {output_path}')
        return stats


print('✓ CachedSearchEngine defined.')

✓ CachedSearchEngine defined.


---
## Part B — Unit Tests + Demo Mode
### B1 — Standalone LRU Eviction Test (no data files needed)

In [27]:
# ── CELL 7: Eviction test — completely standalone, no data needed ─────────────

print('=' * 55)
print('TEST: LRU Eviction (CACHE_MAX_SIZE=5)')
print('=' * 55)

evict_cache = LRUCache(max_size=5)

# ── Insert 6 distinct fake queries ────────────────────────────────────────────
fake_results = [[(i * 100 + j, 0.9 - j * 0.1) for j in range(3)] for i in range(6)]
fake_hashes  = [10, 20, 30, 40, 50, 60]   # distinct integer pHashes

for ph, res in zip(fake_hashes, fake_results):
    evict_cache.put(ph, res)
    print(f'  put(phash={ph})  → cache size = {len(evict_cache)}')

print()
assert len(evict_cache) == 5,            f'Expected 5, got {len(evict_cache)}'
assert evict_cache.get(10) is None,      'phash=10 (oldest) should have been evicted'
assert evict_cache.get(20) is not None,  'phash=20 should still be present'
assert evict_cache.get(60) is not None,  'phash=60 (most recent) should be present'

# ── Access phash=20 to make it MRU, then insert another → phash=30 evicted ──
evict_cache.get(20)       # moves 20 to MRU
evict_cache.put(70, [(700, 0.5)])

assert evict_cache.get(30) is None, 'phash=30 should now be evicted (LRU after 20 was accessed)'
assert evict_cache.get(20) is not None, 'phash=20 should still be present'
print('  Access phash=20 (MRU), insert phash=70 → phash=30 evicted ✓')

# ── Clear test ────────────────────────────────────────────────────────────────
evict_cache.clear()
assert len(evict_cache) == 0, 'cache should be empty after clear()'
print('  clear() → size = 0 ✓')

print()
print('✅ PASS: Eviction test complete.')

TEST: LRU Eviction (CACHE_MAX_SIZE=5)
  put(phash=10)  → cache size = 1
  put(phash=20)  → cache size = 2
  put(phash=30)  → cache size = 3
  put(phash=40)  → cache size = 4
  put(phash=50)  → cache size = 5
  put(phash=60)  → cache size = 5

  Access phash=20 (MRU), insert phash=70 → phash=30 evicted ✓
  clear() → size = 0 ✓

✅ PASS: Eviction test complete.


In [28]:
# ── CELL 8: Approximate match test — standalone, uses only hash values ────────

print('=' * 55)
print('TEST: Approximate Match (Hamming threshold = 8)')
print('=' * 55)

approx_cache = LRUCache(max_size=100)
THRESHOLD    = CACHE_HAMMING_THRESHOLD   # 8

# ── Create a base pHash and a slightly modified one ───────────────────────────
# Flip exactly 4 bits → Hamming distance = 4 (within threshold 8) → approx hit
base_phash     = 0xAABBCCDD11223344     # arbitrary 64-bit integer
close_phash    = base_phash ^ 0xF       # flip lowest 4 bits → distance = 4
far_phash      = base_phash ^ (0xFF << 32)  # flip 8 mid bits → distance = 8  (boundary)
too_far_phash  = base_phash ^ (0x1FF << 32) # flip 9 bits → distance = 9  (miss)

print(f'  base_phash    = {hex(base_phash)}')
print(f'  close_phash   = {hex(close_phash)}  dist = {hamming_distance(base_phash, close_phash)}')
print(f'  far_phash     = {hex(far_phash)}  dist = {hamming_distance(base_phash, far_phash)}')
print(f'  too_far_phash = {hex(too_far_phash)}  dist = {hamming_distance(base_phash, too_far_phash)}')

dummy_result = [(999, 0.95), (998, 0.90)]
approx_cache.put(base_phash, dummy_result)

# close_phash (dist=4) → approximate hit
r1 = approx_cache.find_approximate(close_phash, THRESHOLD)
assert r1 == dummy_result, f'Expected approximate hit for dist=4, got {r1}'
print('  dist=4 → approximate HIT ✓')

# far_phash (dist=8) → hit exactly at boundary
r2 = approx_cache.find_approximate(far_phash, THRESHOLD)
assert r2 == dummy_result, f'Expected approximate hit at boundary dist=8, got {r2}'
print('  dist=8 → approximate HIT (boundary) ✓')

# too_far_phash (dist=9) → miss
r3 = approx_cache.find_approximate(too_far_phash, THRESHOLD)
assert r3 is None, f'Expected MISS for dist=9, got {r3}'
print('  dist=9 → approximate MISS ✓')

print()
print('✅ PASS: Approximate match test complete.')

TEST: Approximate Match (Hamming threshold = 8)
  base_phash    = 0xaabbccdd11223344
  close_phash   = 0xaabbccdd1122334b  dist = 4
  far_phash     = 0xaabbcc2211223344  dist = 8
  too_far_phash = 0xaabbcd2211223344  dist = 9
  dist=4 → approximate HIT ✓
  dist=8 → approximate HIT (boundary) ✓
  dist=9 → approximate MISS ✓

✅ PASS: Approximate match test complete.


### B2 — Demo SearchEngine using test features (600 images)

In [29]:
# ── CELL 9: DemoSearchEngine — uses validation/test_features only ─────────────
# This simulates Mahnoor's SearchEngine interface using only the 600 test images
# that are included in the repo.  It does NOT use the LSH index or the full 1M
# merged feature files.  Suitable for verifying Task 6 logic without 1.5 GB data.

class DemoSearchEngine:
    """
    Minimal SearchEngine-compatible class backed by 600 test images.

    Implements the same interface as Mahnoor's SearchEngine:
        .search(query_embedding, query_image_id, top_k) → list[(id, score)]
        .get_search_stats() → dict
        .query_processor.prepare_query_embedding(image_id) → np.ndarray

    Ranking strategy: cosine similarity on CNN embeddings only.
    This is equivalent to SearchEngine's CNN-only fallback path.
    """

    def __init__(self, test_features_dir: str):
        print('Loading DemoSearchEngine (600 test images) ...')

        cnn_path  = os.path.join(test_features_dir, 'test_cnn.h5')
        hash_path = os.path.join(test_features_dir, 'test_hashes.pkl')

        # CNN embeddings
        with h5py.File(cnn_path, 'r') as f:
            self._embeddings = f['embeddings'][:]      # (600, 128)
            image_ids        = f['image_ids'][:]
        self._id_to_idx = {int(gid): idx for idx, gid in enumerate(image_ids)}
        print(f'  CNN: {self._embeddings.shape[0]} embeddings, dim={self._embeddings.shape[1]}')

        # Normalize all embeddings to unit length once
        norms = np.linalg.norm(self._embeddings, axis=1, keepdims=True)
        norms = np.where(norms < 1e-8, 1e-8, norms)
        self._embeddings_norm = self._embeddings / norms

        # Timing log
        self._search_logs = []

        # Expose a minimal query_processor shim
        self.query_processor = _DemoQueryProcessorShim(self)

        print('✓ DemoSearchEngine ready.')

    def search(self, query_embedding, query_image_id=None, top_k=TOP_K) -> list:
        """Cosine similarity over all 600 test images, returns top_k."""
        t0 = time.time()

        q = np.array(query_embedding, dtype=np.float32).flatten()
        norm = np.linalg.norm(q)
        if norm > 1e-8:
            q = q / norm

        scores = self._embeddings_norm @ q     # (600,)
        ranked = np.argsort(scores)[::-1]      # descending

        # Exclude the query image itself if it is in the test set
        all_ids = list(self._id_to_idx.keys())
        results = []
        for idx in ranked:
            cid = all_ids[idx]
            if cid == query_image_id:
                continue        # skip self-match
            results.append((cid, float(scores[idx])))
            if len(results) == top_k:
                break

        elapsed_ms = (time.time() - t0) * 1000
        self._search_logs.append({'total_ms': elapsed_ms})
        return results

    def get_search_stats(self) -> dict:
        if not self._search_logs:
            return {'mean_latency_ms': 0.0, 'total_queries': 0}
        times = np.array([l['total_ms'] for l in self._search_logs])
        return {
            'total_queries':   len(times),
            'mean_latency_ms': float(np.mean(times)),
            'p50_latency_ms':  float(np.percentile(times, 50)),
            'p95_latency_ms':  float(np.percentile(times, 95)),
        }


class _DemoQueryProcessorShim:
    """Minimal shim so CachedSearchEngine._get_embedding_for_id() works."""
    def __init__(self, engine: DemoSearchEngine):
        self._engine = engine

    def prepare_query_embedding(self, image_id: int = None,
                                raw_embedding = None) -> np.ndarray:
        if image_id is not None:
            if image_id not in self._engine._id_to_idx:
                raise KeyError(f'image_id {image_id} not in test set')
            idx = self._engine._id_to_idx[image_id]
            emb = self._engine._embeddings[idx].astype(np.float32)
        else:
            emb = np.array(raw_embedding, dtype=np.float32).flatten()
        norm = np.linalg.norm(emb)
        return emb / norm if norm > 1e-8 else emb


print('✓ DemoSearchEngine and shim defined.')

✓ DemoSearchEngine and shim defined.


In [30]:
# ── CELL 10: Instantiate Demo CachedSearchEngine ──────────────────────────────

# Load test pHash lookup (600 test images, IDs 2,000,000 – 2,000,599)
test_hash_path = os.path.join(TEST_FEATURES_DIR, 'test_hashes.pkl')
with open(test_hash_path, 'rb') as f:
    test_phash_lookup = pickle.load(f)

print(f'Loaded {len(test_phash_lookup)} test pHash entries.')
sample_id  = list(test_phash_lookup.keys())[0]
print(f'Sample entry — ID {sample_id}: {test_phash_lookup[sample_id]}')
print()

# Build the demo engine
demo_engine = DemoSearchEngine(TEST_FEATURES_DIR)

# Wrap with the caching layer (same max_size and threshold as full system)
cached_engine = CachedSearchEngine(
    search_engine  = demo_engine,
    phash_lookup   = test_phash_lookup,
    max_size       = CACHE_MAX_SIZE,
    hamming_thresh = CACHE_HAMMING_THRESHOLD
)

Loaded 600 test pHash entries.
Sample entry — ID 2000000: {'ahash': 3600486611316113408, 'dhash': 13962523170636884400, 'phash': 10585548261292710187}

Loading DemoSearchEngine (600 test images) ...
  CNN: 600 embeddings, dim=128
✓ DemoSearchEngine ready.
✓ CachedSearchEngine ready.  Cache size=1000, hamming_thresh=8


In [31]:
# ── CELL 11: Test 1 — Exact hit test ─────────────────────────────────────────

print('=' * 55)
print('TEST 1: Exact hit — query same image_id twice')
print('=' * 55)

TEST_ID = 2000000

# ── First query → cache miss ──────────────────────────────────────────────────
t0 = time.perf_counter()
results_first = cached_engine.cached_search(query_image_id=TEST_ID, top_k=10)
t1 = time.perf_counter()
first_ms = (t1 - t0) * 1000

print(f'First call  (miss ) : {first_ms:.2f} ms  |  '
      f'misses={cached_engine.cache_misses}')
print(f'  Results: {results_first[:3]} ...')

# ── Second query (same ID) → exact cache hit ──────────────────────────────────
t2 = time.perf_counter()
results_second = cached_engine.cached_search(query_image_id=TEST_ID, top_k=10)
t3 = time.perf_counter()
second_ms = (t3 - t2) * 1000

print(f'Second call (hit  ) : {second_ms:.3f} ms  |  '
      f'exact_hits={cached_engine.exact_hits}')

assert results_first == results_second,  'Cache hit must return identical results'
assert cached_engine.exact_hits == 1,   f'Expected exact_hits=1, got {cached_engine.exact_hits}'
assert second_ms < first_ms,            f'Cache hit ({second_ms:.3f}ms) should be faster than miss ({first_ms:.2f}ms)'

print()
print(f'  Speedup : {first_ms / second_ms:.1f}×  '
      f'({first_ms:.2f}ms → {second_ms:.3f}ms)')
print('✅ PASS: Exact hit test.')

TEST 1: Exact hit — query same image_id twice
First call  (miss ) : 12.67 ms  |  misses=1
  Results: [(2000005, 0.9734537601470947), (2000001, 0.9681147336959839), (2000004, 0.9557126760482788)] ...
Second call (hit  ) : 0.091 ms  |  exact_hits=1

  Speedup : 138.7×  (12.67ms → 0.091ms)
✅ PASS: Exact hit test.


In [32]:
# ── CELL 12: Test 2 — Approximate hit test ────────────────────────────────────
# Uses ground_truth.json: base images and their near-duplicate variants.
# Variants with Hamming distance ≤ 8 from the base should be approx. cache hits.

print('=' * 55)
print('TEST 2: Approximate hit (brightness-variant images)')
print('=' * 55)

with open(GROUND_TRUTH_PATH) as f:
    ground_truth = json.load(f)

# Find a base-variant pair whose pHash Hamming distance is within the threshold
approx_test_pair = None
for base_id_str, variant_ids in ground_truth.items():
    base_id   = int(base_id_str)
    base_ph   = test_phash_lookup[base_id]['phash']
    for v_id in variant_ids:
        if v_id in test_phash_lookup:
            d = hamming_distance(base_ph, test_phash_lookup[v_id]['phash'])
            if 1 <= d <= CACHE_HAMMING_THRESHOLD:  # distance > 0 (not exact) but within threshold
                approx_test_pair = (base_id, v_id, d)
                break
    if approx_test_pair:
        break

if approx_test_pair is None:
    print('No pair found within threshold — checking test hashes ...')
    # Fallback: create a pair by flipping bits manually
    base_id  = 2000000
    base_ph  = test_phash_lookup[base_id]['phash']
    approx_ph = base_ph ^ 0b1111   # flip 4 bits
    print('  Using synthetic pair (manual bit flip)')
else:
    base_id, variant_id, dist = approx_test_pair
    print(f'Found pair: base={base_id}, variant={variant_id}, '
          f'pHash Hamming dist={dist}')

# Reset counters for a clean test
approx_engine = CachedSearchEngine(
    search_engine  = demo_engine,
    phash_lookup   = test_phash_lookup,
    max_size       = 100,
    hamming_thresh = CACHE_HAMMING_THRESHOLD
)

# Step 1: Query the BASE image → cache miss, result stored
base_id, variant_id, dist = approx_test_pair
r_base = approx_engine.cached_search(query_image_id=base_id, top_k=10)
print(f'\nQueried base   ID={base_id} → MISS  (misses={approx_engine.cache_misses})')

# Step 2: Query the VARIANT image → should be approximate cache hit
r_variant = approx_engine.cached_search(query_image_id=variant_id, top_k=10)
print(f'Queried variant ID={variant_id} → '
      f'approx_hits={approx_engine.approximate_hits}')

assert approx_engine.approximate_hits >= 1, (
    f'Expected at least 1 approximate hit, got {approx_engine.approximate_hits}. '
    f'Hamming dist={dist}, threshold={CACHE_HAMMING_THRESHOLD}')

# The variant result should equal the base result (from cache)
assert r_base == r_variant, 'Approximate cache hit must return same results as base'

# Query the variant AGAIN → now an exact hit (its phash was registered in step 2)
r_variant2 = approx_engine.cached_search(query_image_id=variant_id, top_k=10)
assert approx_engine.exact_hits >= 1, 'Second variant query should be exact hit'
print(f'Queried variant again → exact_hits={approx_engine.exact_hits}')

print()
print('✅ PASS: Approximate hit test.')

TEST 2: Approximate hit (brightness-variant images)
Found pair: base=2000000, variant=2000003, pHash Hamming dist=4
✓ CachedSearchEngine ready.  Cache size=100, hamming_thresh=8

Queried base   ID=2000000 → MISS  (misses=1)
Queried variant ID=2000003 → approx_hits=1
Queried variant again → exact_hits=1

✅ PASS: Approximate hit test.


In [33]:
# ── CELL 13: Integration test — 20 queries with cache statistics ──────────────

print('=' * 55)
print('INTEGRATION TEST: 20 queries over ground truth')
print('=' * 55)

# Fresh engine for clean stats
int_engine = CachedSearchEngine(
    search_engine  = demo_engine,
    phash_lookup   = test_phash_lookup,
    max_size       = CACHE_MAX_SIZE,
    hamming_thresh = CACHE_HAMMING_THRESHOLD
)

test_base_ids = [int(k) for k in list(ground_truth.keys())[:10]]

# ── Phase 1: Query each base image (all cache misses) ────────────────────────
print('\nPhase 1 — 10 base image queries (expect all misses):')
for qid in test_base_ids:
    results = int_engine.cached_search(query_image_id=qid, top_k=10)
    print(f'  ID={qid}  → top result: {results[0] if results else "none"}')

print(f'\n  After phase 1: misses={int_engine.cache_misses}, '
      f'cache_size={len(int_engine.cache)}')

# ── Phase 2: Re-query the same IDs + near-duplicate variants ─────────────────
print('\nPhase 2 — re-query same 10 IDs (expect all exact hits):')
for qid in test_base_ids:
    results = int_engine.cached_search(query_image_id=qid, top_k=10)

print(f'  After phase 2: exact_hits={int_engine.exact_hits}')
assert int_engine.exact_hits == 10, f'Expected 10 exact hits, got {int_engine.exact_hits}'

# ── Phase 3: Query near-duplicate variants ────────────────────────────────────
print('\nPhase 3 — query near-duplicate variants (expect approx hits where dist≤8):')
approx_count_before = int_engine.approximate_hits
for base_id_str, variant_ids in list(ground_truth.items())[:10]:
    for v_id in variant_ids:
        if v_id in test_phash_lookup:
            int_engine.cached_search(query_image_id=v_id, top_k=10)

print(f'  Approximate hits found: '
      f'{int_engine.approximate_hits - approx_count_before}')

# ── Final stats ───────────────────────────────────────────────────────────────
print()
stats = int_engine.get_cache_stats()
print('─' * 45)
print('CACHE STATISTICS')
print('─' * 45)
for k, v in stats.items():
    print(f'  {k:<30}: {v}')

assert stats['hit_rate_pct'] > 0,   'Hit rate should be > 0 after phase 2 hits'
print()
print('✅ PASS: Integration test complete.')

INTEGRATION TEST: 20 queries over ground truth
✓ CachedSearchEngine ready.  Cache size=1000, hamming_thresh=8

Phase 1 — 10 base image queries (expect all misses):
  ID=2000000  → top result: (2000005, 0.9734537601470947)
  ID=2000006  → top result: (2000007, 0.9870281219482422)
  ID=2000012  → top result: (2000016, 0.9231194257736206)
  ID=2000018  → top result: (2000023, 0.9705423712730408)
  ID=2000024  → top result: (2000027, 0.9631725549697876)
  ID=2000030  → top result: (2000035, 0.9620412588119507)
  ID=2000036  → top result: (2000038, 0.9603321552276611)
  ID=2000042  → top result: (2000045, 0.9689834117889404)
  ID=2000048  → top result: (2000052, 0.9711552858352661)
  ID=2000054  → top result: (2000058, 0.9369348287582397)

  After phase 1: misses=10, cache_size=10

Phase 2 — re-query same 10 IDs (expect all exact hits):
  After phase 2: exact_hits=10

Phase 3 — query near-duplicate variants (expect approx hits where dist≤8):
  Approximate hits found: 18

───────────────────

In [34]:
# ── CELL 14: Cache warm-up test ───────────────────────────────────────────────

print('=' * 55)
print('TEST: warm_cache() — pre-populate with 5 IDs')
print('=' * 55)

warm_engine = CachedSearchEngine(
    search_engine  = demo_engine,
    phash_lookup   = test_phash_lookup,
    max_size       = 50,
    hamming_thresh = CACHE_HAMMING_THRESHOLD
)

warm_ids = test_base_ids[:5]
warm_engine.warm_cache(warm_ids, top_k=10)

assert len(warm_engine.cache) == 5, f'Expected 5 cached entries, got {len(warm_engine.cache)}'

# Querying a warm ID should be an exact hit now
warm_engine.total_queries   = 0   # reset to measure cleanly
warm_engine.cache_hits      = 0
warm_engine.exact_hits      = 0
warm_engine.cache_misses    = 0

for wid in warm_ids:
    warm_engine.cached_search(query_image_id=wid, top_k=10)

assert warm_engine.exact_hits == 5, f'Expected 5 exact hits, got {warm_engine.exact_hits}'
print(f'All {len(warm_ids)} warm IDs returned as exact hits ✓')
print()
print('✅ PASS: warm_cache() test.')

TEST: warm_cache() — pre-populate with 5 IDs
✓ CachedSearchEngine ready.  Cache size=50, hamming_thresh=8
Warming cache with 5 images ...


Warm-up: 100%|██████████| 5/5 [00:00<00:00, 3278.85img/s]

Cache warmed.  Current size: 5
All 5 warm IDs returned as exact hits ✓

✅ PASS: warm_cache() test.


In [35]:
# ── CELL 15: Save cache_stats.json ────────────────────────────────────────────

saved_stats = int_engine.save_cache_stats(
    os.path.join(CACHE_OUTPUT_DIR, 'cache_stats.json')
)

print()
print('Saved JSON preview:')
print(json.dumps(saved_stats, indent=2))

✓ Cache stats saved to /kaggle/working/cache/cache_stats.json

Saved JSON preview:
{
  "total_queries": 70,
  "cache_hits": 43,
  "cache_misses": 27,
  "exact_hits": 25,
  "approximate_hits": 18,
  "cache_bypasses": 0,
  "hit_rate_pct": 61.43,
  "current_cache_size": 45,
  "max_cache_size": 1000,
  "mean_search_latency_ms": 0.49,
  "estimated_time_saved_ms": 21.01
}


---
## Part C — Full System Integration
*(Requires all merged M1 feature files in `data/` — run locally only)*

In [8]:
# ── CELL 16: Full SearchEngine integration ────────────────────────────────────
# Connects to Mahnoor's SearchEngine and tests the full pipeline.
# Only runs if all merged M1 feature files are present in data/.

if not FULL_DATA_AVAILABLE:
    print('⚠️  SKIPPING full integration — merged M1 feature files not found in data/.')
    print('    Place the following files in data/ and re-run:')
    for fname in ['all_cnn_embeddings.h5', 'all_perceptual_hashes.pkl',
                  'all_sift_features.pkl', 'all_orb_features.pkl',
                  'all_hist_features.pkl', 'all_image_paths.pkl']:
        fpath = os.path.join(DATA_DIR, fname)
        status = '✅' if os.path.exists(fpath) else '❌'
        print(f'    {status} {fname}')
else:
    print('✅ Full data available — loading SearchEngine ...')
    print('   (This takes ~60s and needs ~1.5 GB RAM)')

    from query_engine.search_engine import SearchEngine

    full_engine = SearchEngine(
        index_dir = INDEX_DIR,
        data_dir  = DATA_DIR,
    )

    # Load the full perceptual hash lookup (all 1M images)
    with open(os.path.join(DATA_DIR, 'all_perceptual_hashes.pkl'), 'rb') as f:
        full_phash_lookup = pickle.load(f)
    print(f'Full phash lookup: {len(full_phash_lookup):,} entries')

    full_cached = CachedSearchEngine(
        search_engine  = full_engine,
        phash_lookup   = full_phash_lookup,
        max_size       = CACHE_MAX_SIZE,
        hamming_thresh = CACHE_HAMMING_THRESHOLD
    )

    # ── Replace from "Running 10 queries..." to end of Cell 16 ───────────────────

    # ground_truth.json uses test IDs (2000000+) which are NOT in the 1M index.
    # For the full system test we pick 10 random IDs from the actual indexed dataset.
    import random
    all_indexed_ids = list(full_engine.feature_store.id_to_index.keys())
    random.seed(42)
    test_ids = random.sample(all_indexed_ids, 10)

    print(f'Running 10 queries on indexed IDs: {test_ids}\n')

    hits_at_10     = 0
    total_results  = 0

    for qid in test_ids:
        t0      = time.perf_counter()
        results = full_cached.cached_search(query_image_id=qid, top_k=10)
        ms      = (time.perf_counter() - t0) * 1000

        print(f'  ID={qid:>9}  {ms:>7.1f}ms  top result: {results[0] if results else "none"}')
        total_results += len(results)

    # ── Re-run same IDs → all should be exact cache hits ─────────────────────────
    print('\nRe-running same 10 queries (expect exact cache hits):')
    for qid in test_ids:
        t0      = time.perf_counter()
        results = full_cached.cached_search(query_image_id=qid, top_k=10)
        ms      = (time.perf_counter() - t0) * 1000
        print(f'  ID={qid:>9}  {ms:>6.3f}ms (cached)')

    # ── Stats + save final cache_stats.json ──────────────────────────────────────
    final_stats = full_cached.save_cache_stats(
        os.path.join(CACHE_OUTPUT_DIR, 'cache_stats.json')
    )

    print()
    print('─' * 45)
    print('FINAL CACHE STATISTICS')
    print('─' * 45)
    for k, v in final_stats.items():
        print(f'  {k:<35}: {v}')

    assert full_cached.exact_hits == 10, \
        f'Expected 10 exact hits on re-run, got {full_cached.exact_hits}'

    print()
    print('✅ Part C complete.')
    print(f'   cache_stats.json saved to: {CACHE_OUTPUT_DIR}')
    print('   → Upload this file to PDC_Project_Shared/cache/ on Google Drive')

✅ Full data available — loading SearchEngine ...
   (This takes ~60s and needs ~1.5 GB RAM)
Initialising QueryProcessor (Task 3) ...
Config loaded: 10 tables, hash_size=12, max_candidates=5000
Projection matrices loaded: 10 × (12, 128)
Loading 10 hash tables from C:\Users\USER\Desktop\pdc proj\pdc\PDC_distributed-reverse-image-search\lsh_index\hash_tables ...
  table_0: 4,096 buckets, 1,000,000 image ID entries
  table_1: 4,096 buckets, 1,000,000 image ID entries
  table_2: 4,096 buckets, 1,000,000 image ID entries
  table_3: 4,096 buckets, 1,000,000 image ID entries
  table_4: 4,096 buckets, 1,000,000 image ID entries
  table_5: 4,096 buckets, 1,000,000 image ID entries
  table_6: 4,096 buckets, 1,000,000 image ID entries
  table_7: 4,096 buckets, 1,000,000 image ID entries
  table_8: 4,096 buckets, 1,000,000 image ID entries
  table_9: 4,096 buckets, 1,000,000 image ID entries
All tables loaded in 2.0s

Building id→index mapping from HDF5 ...
  Mapping built: 1,000,000 entries

Initi

---
## Part D — Export `query_cache.py` Module

In [9]:
# ── FIX CELL: Write correct query_cache.py ───────────────────────────────────

output_path2 = r'C:\Users\USER\Desktop\pdc proj\pdc\PDC_distributed-reverse-image-search\cache\query_cache.py'

code = """\
\"\"\"
query_cache.py — Task 6: Query Caching Layer
Project 20: Distributed Reverse Image Search Engine — Milestone 2
\"\"\"

import os, json, time, collections
import numpy as np
from PIL import Image
import imagehash
from tqdm import tqdm

_CFG_PATH = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))),
                         'lsh_index', 'lsh_config.json')
with open(_CFG_PATH) as _f:
    _CFG = json.load(_f)

CACHE_MAX_SIZE          = _CFG['cache_max_size']
CACHE_HAMMING_THRESHOLD = _CFG['cache_hamming_threshold']
TOP_K                   = _CFG['top_k']
HASH_SIZE               = 8


def hamming_distance(h1: int, h2: int) -> int:
    return bin(h1 ^ h2).count('1')


def hash_similarity(h1: int, h2: int) -> float:
    return 1.0 - (hamming_distance(h1, h2) / 64.0)


class LRUCache:
    def __init__(self, max_size: int):
        if max_size < 1:
            raise ValueError('max_size must be >= 1')
        self._max_size = max_size
        self._store    = collections.OrderedDict()

    def get(self, phash_key: int):
        if phash_key not in self._store:
            return None
        result_list, _ = self._store[phash_key]
        self._store.move_to_end(phash_key)
        self._store[phash_key] = (result_list, time.time())
        return result_list

    def put(self, phash_key: int, result_list: list):
        if phash_key in self._store:
            self._store.move_to_end(phash_key)
        self._store[phash_key] = (result_list, time.time())
        if len(self._store) > self._max_size:
            self._store.popitem(last=False)

    def find_approximate(self, query_phash: int, threshold: int):
        best_key, best_dist = None, threshold + 1
        for cached_key, (result_list, _) in reversed(self._store.items()):
            dist = hamming_distance(query_phash, cached_key)
            if dist < best_dist:
                best_dist, best_key = dist, cached_key
        if best_key is not None and best_dist <= threshold:
            result_list, _ = self._store[best_key]
            self._store.move_to_end(best_key)
            self._store[best_key] = (result_list, time.time())
            return result_list
        return None

    def clear(self):
        self._store.clear()

    def __len__(self):
        return len(self._store)

    def __repr__(self):
        return f'LRUCache(size={len(self)}/{self._max_size})'


class CachedSearchEngine:
    def __init__(self, search_engine, phash_lookup: dict,
                 max_size: int = CACHE_MAX_SIZE,
                 hamming_thresh: int = CACHE_HAMMING_THRESHOLD):
        self.search_engine    = search_engine
        self.phash_lookup     = phash_lookup
        self.hamming_thresh   = hamming_thresh
        self.cache            = LRUCache(max_size=max_size)
        self.total_queries    = 0
        self.cache_hits       = 0
        self.exact_hits       = 0
        self.approximate_hits = 0
        self.cache_misses     = 0
        self.cache_bypasses   = 0
        print(f'✓ CachedSearchEngine ready. Cache={max_size}, thresh={hamming_thresh}')

    def cached_search(self, query_image_id=None, query_embedding=None,
                      query_pil_image=None, top_k: int = TOP_K) -> list:
        n = sum(x is not None for x in [query_image_id, query_embedding, query_pil_image])
        if n != 1:
            raise ValueError('Provide exactly one input type.')
        self.total_queries += 1

        if query_embedding is not None:
            self.cache_bypasses += 1
            emb  = np.array(query_embedding, dtype=np.float32).flatten()
            norm = np.linalg.norm(emb)
            emb  = emb / norm if norm > 1e-8 else emb
            return self.search_engine.search(query_embedding=emb,
                                             query_image_id=None, top_k=top_k)

        if query_pil_image is not None:
            ph = int(str(imagehash.phash(query_pil_image, hash_size=HASH_SIZE)), 16)
            return self._lookup_and_search(ph,
                       lambda: self._embed_pil(query_pil_image), None, top_k)

        if query_image_id not in self.phash_lookup:
            raise KeyError(f'image_id {query_image_id} not in phash_lookup.')
        ph = self.phash_lookup[query_image_id]['phash']
        return self._lookup_and_search(ph,
                   lambda: self._get_embedding_for_id(query_image_id),
                   query_image_id, top_k)

    def _lookup_and_search(self, query_phash, embedding_fn, image_id, top_k):
        cached = self.cache.get(query_phash)
        if cached is not None:
            self.cache_hits += 1
            self.exact_hits += 1
            return cached
        approx = self.cache.find_approximate(query_phash, self.hamming_thresh)
        if approx is not None:
            self.cache_hits       += 1
            self.approximate_hits += 1
            self.cache.put(query_phash, approx)
            return approx
        self.cache_misses += 1
        result = self.search_engine.search(query_embedding=embedding_fn(),
                                           query_image_id=image_id, top_k=top_k)
        self.cache.put(query_phash, result)
        return result

    def _get_embedding_for_id(self, image_id):
        return self.search_engine.query_processor.prepare_query_embedding(
            image_id=image_id)

    def _embed_pil(self, pil_image):
        raise NotImplementedError(
            'PIL embedding extraction requires a CNN model. '
            'Use query_embedding= instead.')

    def warm_cache(self, image_ids: list, top_k: int = TOP_K):
        print(f'Warming cache with {len(image_ids)} images ...')
        for iid in tqdm(image_ids, desc='Warm-up', unit='img'):
            try:
                self.cached_search(query_image_id=iid, top_k=top_k)
            except Exception as e:
                print(f'  Skipped {iid}: {e}')

    def get_cache_stats(self) -> dict:
        hit_rate = (self.cache_hits / self.total_queries * 100.0
                    if self.total_queries > 0 else 0.0)
        try:
            mean_ms = self.search_engine.get_search_stats().get('mean_latency_ms', 0.0)
        except Exception:
            mean_ms = 0.0
        return {
            'total_queries':           self.total_queries,
            'cache_hits':              self.cache_hits,
            'cache_misses':            self.cache_misses,
            'exact_hits':              self.exact_hits,
            'approximate_hits':        self.approximate_hits,
            'cache_bypasses':          self.cache_bypasses,
            'hit_rate_pct':            round(hit_rate, 2),
            'current_cache_size':      len(self.cache),
            'max_cache_size':          self.cache._max_size,
            'mean_search_latency_ms':  round(mean_ms, 2),
            'estimated_time_saved_ms': round(self.cache_hits * mean_ms, 2),
        }

    def save_cache_stats(self, output_path2: str = None):
        if output_path2 is None:
            output_path2 = os.path.join(
                os.path.dirname(os.path.abspath(__file__)), 'cache_stats.json')
        stats = self.get_cache_stats()
        with open(output_path2, 'w') as f:
            json.dump(stats, f, indent=2)
        print(f'✓ Cache stats → {output_path2}')
        return stats
"""

with open(output_path2, 'w') as f:
    f.write(code)

print(f'✓ query_cache.py written ({os.path.getsize(output_path2):,} bytes)')
print(f'  Path: {output_path2}')

✓ query_cache.py written (7,469 bytes)
  Path: C:\Users\USER\Desktop\pdc proj\pdc\PDC_distributed-reverse-image-search\cache\query_cache.py


In [40]:
# ── CELL 18: Final summary ────────────────────────────────────────────────────

print('=' * 60)
print('TASK 6 — COMPLETE SUMMARY')
print('=' * 60)
print()
print('Files produced:')
print(f'  ✅  {CACHE_MODULE_PATH}')
print(f'  ✅  {os.path.join(CACHE_OUTPUT_DIR, "cache_stats.json")}')
print()
print('Tests passed:')
print('  ✅  LRU eviction (CACHE_MAX_SIZE=5, 6 insertions)')
print('  ✅  Approximate match (Hamming ≤ 8 bits → cache hit)')
print('  ✅  Exact hit (same ID twice — second call <1 ms)')
print('  ✅  warm_cache() — pre-populates then all exact hits')
print('  ✅  Integration test — 20 queries, hit/miss/approx stats')
print()
print('To run the FULL system (Part C):')
print('  1. Download all merged M1 files to data/')
print('     (all_cnn_embeddings.h5, all_perceptual_hashes.pkl, etc.)')
print('  2. Confirm lsh_index/hash_tables/ has Piranchal\'s full 1M index')
print('  3. Re-run Cell 16 — it will auto-detect and load SearchEngine')
print()
print('Why CANNOT run on Kaggle (full mode):')
print('  • Merged M1 feature files (~1.5 GB) are on Google Drive,')
print('    not as public Kaggle datasets')
print('  • The full LSH index was built by Piranchal locally')
print('  • All these files must be present locally for SearchEngine')
print('  • Demo mode (600 test images) DOES work on Kaggle')
print('=' * 60)

TASK 6 — COMPLETE SUMMARY

Files produced:
  ✅  /kaggle/working/cache/query_cache.py
  ✅  /kaggle/working/cache/cache_stats.json

Tests passed:
  ✅  LRU eviction (CACHE_MAX_SIZE=5, 6 insertions)
  ✅  Approximate match (Hamming ≤ 8 bits → cache hit)
  ✅  Exact hit (same ID twice — second call <1 ms)
  ✅  warm_cache() — pre-populates then all exact hits
  ✅  Integration test — 20 queries, hit/miss/approx stats

To run the FULL system (Part C):
  1. Download all merged M1 files to data/
     (all_cnn_embeddings.h5, all_perceptual_hashes.pkl, etc.)
  2. Confirm lsh_index/hash_tables/ has Piranchal's full 1M index
  3. Re-run Cell 16 — it will auto-detect and load SearchEngine

Why CANNOT run on Kaggle (full mode):
  • Merged M1 feature files (~1.5 GB) are on Google Drive,
    not as public Kaggle datasets
  • The full LSH index was built by Piranchal locally
  • All these files must be present locally for SearchEngine
  • Demo mode (600 test images) DOES work on Kaggle
